Get all relevant EKP nan samples

In [ ]:
import os
import pandas as pd

csv_datei = "/groups/ds/Win-KID/BVBRC/BVBRC_genome_amr_02.2025.csv"
df = pd.read_csv(csv_datei, sep=",")

nan_rows = df[df["Laboratory Typing Platform"].isna()]
print(
    f"Unique genome IDs with NaN in 'Laboratory Typing Platform': {len(nan_rows["Genome ID"].unique())}"
)
non_nan_rows = df[~df["Laboratory Typing Platform"].isna()]

# Use only samples that are not in devices dataset
nan_rows = nan_rows[~nan_rows["Genome ID"].isin(non_nan_rows["Genome ID"])]

# Filter out EKP
nan_rows["Species"] = (
    nan_rows["Genome Name"].astype(str).str.split().str[:2].str.join(" ")
)
nan_EKP_df = nan_rows[
    nan_rows["Species"].str.contains("Klebsiella pneumoniae", case=False, na=False)
]

# Filter out samples with fna
fna_folder = "/groups/ds/Win-KID/BVBRC/fna"
existing_ids = [
    gid
    for gid in nan_EKP_df["Genome ID"].unique()
    if os.path.exists(os.path.join(fna_folder, f"{gid}.fna.gz"))
]

print("Number of existing_ids nan EKP Genomes:")
print(len(existing_ids))
existing_ids_df = pd.DataFrame(existing_ids, columns=["Genome ID"])
existing_ids_df.to_csv("exisisting_nan_ids.csv", index=False)

nan_EKP_df = nan_EKP_df[nan_EKP_df["Genome ID"].isin(existing_ids_df["Genome ID"])]

nan_EKP_df.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/existing_na.csv", index=False
)

/local/tmp/ipykernel_3854037/633084981.py:5: DtypeWarning: Columns (0: Computational Method Version, 1: Source) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_datei, sep=",")


Unique genome IDs with NaN in 'Laboratory Typing Platform': 179634
Number of existing_ids nan EKP Genomes:
22333


Filter out usefull samples with more than six resistance results

In [6]:
import pandas as pd

interpreted_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/interpreted/exisiting_EKP_nan_interpreted.csv"
)
print(interpreted_df)
interpreted_df = interpreted_df.dropna(subset=interpreted_df.columns[2:], how="all")
interpreted_df = interpreted_df.replace(["NA", "", " "], pd.NA)
interpreted_df = interpreted_df[interpreted_df.isna().sum(axis=1) <= 30]

print(len(interpreted_df))
print(interpreted_df.notna().sum(axis=1).mean())
interpreted_df.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/EKP_na_interpreted.csv", index=False
)

       Sample_ID_IfH Organism_Code imipenem amikacin gentamicin cefepime  \
0          573.35474           EKP      NaN      NaN        NaN      NaN   
1          573.21472           EKP      NaN      NaN        NaN      NaN   
2          573.16457           EKP      NaN      NaN        NaN      NaN   
3          573.23149           EKP      NaN      NaN        NaN      NaN   
4          573.36094           EKP      NaN      NaN        NaN      NaN   
...              ...           ...      ...      ...        ...      ...   
22328      573.96320           EKP      NaN      NaN        NaN      NaN   
22329      573.70040           EKP      NaN      NaN        NaN      NaN   
22330      573.96310           EKP      NaN      NaN        NaN      NaN   
22331      573.97030           EKP      NaN      NaN        NaN      NaN   
22332      573.96430           EKP      NaN      NaN        NaN      NaN   

      meropenem levofloxacin trimethoprim-sulfamethoxazole ertapenem  ...  \
0         

/local/tmp/ipykernel_3888767/480222167.py:3: DtypeWarning: Columns (0: imipenem-relebactam, 1: ceftaroline, 2: cefiderocol, 3: sulfamethoxazole) have mixed types. Specify dtype option on import or set low_memory=False.
  interpreted_df = pd.read_csv(


Cross validate EKP NA values
Random select EKP Phoenix values 
Random select EKP VITEK values CV

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Clean up NA
df_na = pd.read_csv("EKP_na_interpreted_bin.csv")
df_na = df_na.dropna(subset=df_na.columns[2:], how="all")

# MIN_SAMPLE_NUMBER = 15
df_na_cleanup = pd.read_csv("samples_used_NA_15.csv")
df_na_cleaned = df_na[df_na["Sample_ID_IfH"].isin(df_na_cleanup["Sample_ID_IfH"])]

df_na_cleaned.to_csv("na_cleaned.csv", index=False)
print("mean na entries")
print(df_na_cleaned.notna().sum(axis=1).mean())
# Clean up VITEK
df_vitek = pd.read_csv("VITEK_combined_02_26_interpreted_bin.csv")
df_vitek = df_vitek.dropna(subset=df_vitek.columns[2:], how="all")
df_vitek_EKP = df_vitek[df_vitek["Organism_Code"] == "EKP"]
df_vitek_cleanup = pd.read_csv("cleanup/samples_used_VITEK_15.csv")
df_vitek_cleaned = df_vitek_EKP[
    df_vitek_EKP["Sample_ID_IfH"].isin(df_vitek_cleanup["Sample_ID_IfH"])
]

df_vitek_cleaned.to_csv("VITEK_cleaned.csv", index=False)
print("mean vitek entries")
print(df_vitek_cleaned.notna().sum(axis=1).mean())

# Clean up phoenix
df_phoenix = pd.read_csv("phoenix_combined_02_26_interpreted_bin.csv")
df_phoenix_EKP = df_phoenix[df_phoenix["Organism_Code"] == "EKP"]
df_phoenix_EKP = df_phoenix_EKP.dropna(subset=df_phoenix_EKP.columns[2:], how="all")
df_phoenix_cleanup = pd.read_csv("samples_used_phoenix_15.csv")
df_phoenix_cleaned = df_phoenix_EKP[
    df_phoenix_EKP["Sample_ID_IfH"].isin(df_phoenix_cleanup["Sample_ID_IfH"])
]

df_phoenix_cleaned.to_csv("phoenix_cleaned.csv", index=False)
print("mean phoenix entries")
print(df_phoenix_cleaned.notna().sum(axis=1).mean())

mean na entries
14.442534908700322
mean vitek entries
21.193396226415093
mean phoenix entries
18.878730158730157


Create and save splits

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold
from pathlib import Path

N_SPLITS = 5
N_REPEATS = 1
OUTPUT_DIR = Path("")

na_df = pd.read_csv("na_cleaned.csv")
vitek_df = pd.read_csv("VITEK_cleaned.csv")
phoenix_df = pd.read_csv("phoenix_cleaned.csv")

remainder = len(na_df) % N_SPLITS
if remainder > 0:
    na_df = na_df.sample(n=len(na_df) - remainder, random_state=42).reset_index(
        drop=True
    )
PHOENIX_SUBSAMPLE = int(len(na_df) / N_SPLITS)
VITEK_SUBSAMPLE = len(vitek_df) - PHOENIX_SUBSAMPLE

for repeat in range(N_REPEATS):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=repeat)

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(na_df)):
        seed = repeat * 100 + fold_idx
        run_name = f"repeat{repeat}_fold{fold_idx}"

        run_dir = OUTPUT_DIR / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        na_train = na_df.iloc[test_idx]
        na_test = na_df.iloc[train_idx]

        # Random sampling
        vitek_sample = vitek_df.sample(n=VITEK_SUBSAMPLE, random_state=seed)

        phoenix_sample = phoenix_df.sample(n=PHOENIX_SUBSAMPLE, random_state=seed)

        # Save
        na_train.to_csv(run_dir / "na_train.csv", index=False)
        na_test.to_csv(run_dir / "na_test.csv", index=False)

        vitek_sample.to_csv(run_dir / "vitek_train.csv", index=False)
        phoenix_sample.to_csv(run_dir / "phoenix_train.csv", index=False)

        print(f"Saved: {run_dir}")

Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/repeat0_fold0
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/repeat0_fold1
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/repeat0_fold2
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/repeat0_fold3
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/repeat0_fold4


Generate DataSet counts

In [1]:
import pandas as pd

na_df = pd.read_csv("na_cleaned.csv")
vitek_df = pd.read_csv("VITEK_cleaned.csv")
phoenix_df = pd.read_csv("phoenix_cleaned.csv")


def count_s_r_per_antibiotic(df: pd.DataFrame) -> pd.DataFrame:
    # Identify antibiotic columns
    antibiotic_cols = [
        col for col in df.columns if col not in ["Sample_ID_IfH", "Organism_Code"]
    ]

    # Count S and R for each antibiotic
    sr_counts = pd.DataFrame(
        {
            "S": (df[antibiotic_cols] == "S").sum(),
            "R": (df[antibiotic_cols] == "R").sum(),
        }
    )

    # Calculate total number of available results
    sr_counts["Total"] = sr_counts["S"] + sr_counts["R"]

    # Remove antibiotics without any S or R result
    sr_counts = sr_counts[sr_counts["Total"] > 0]

    # Convert antibiotic names from index to column
    sr_counts = sr_counts.reset_index()
    sr_counts = sr_counts.rename(columns={"index": "Antibiotic"})

    return sr_counts


vitek_count = count_s_r_per_antibiotic(vitek_df)
print("VITEK")
print(vitek_count)
vitek_count.to_csv("vitek_count.csv", index=False)

phoenix_count = count_s_r_per_antibiotic(phoenix_df)
print("phoenix")
print(phoenix_count)
phoenix_count.to_csv("phoenix_count.csv", index=False)

na_count = count_s_r_per_antibiotic(na_df)
print("NA")
print(na_count)
na_count.to_csv("na_count.csv", index=False)

VITEK
                       Antibiotic    S    R  Total
0                       meropenem  295  553    848
1                       ertapenem  230  580    810
2                      tobramycin  266  426    692
3                     tigecycline   18  665    683
4         piperacillin-tazobactam  226  467    693
5                      cefotaxime   40  672    712
6                     cefpodoxime    0  468    468
7     amoxicillin-clavulanic acid   39  195    234
8                      ampicillin   27  759    786
9                      gentamicin  426  422    848
10                       amikacin  618  230    848
11                    ceftriaxone   16  526    542
12                       cefepime   99  593    692
13                   trimethoprim   25   54     79
14  trimethoprim-sulfamethoxazole    0  751    751
15                  ciprofloxacin   84  764    848
16                    ceftazidime  245  492    737
17               sulfamethoxazole    0   10     10
18                      a